In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('/Healthcare_Cohort.csv')

In [ ]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'])

In [ ]:
df.head()

,Order_ID,Customer_ID,Order_Date,Product,Category,Revenue
0,100001,C000002,2025-01-19,Immunity Booster,Ayurvedic,1578
1,100002,C000003,2025-01-14,Immunity Booster,Ayurvedic,358
2,100003,C000005,2025-01-23,Immunity Booster,Ayurvedic,1627
3,100004,C000006,2025-01-25,Joint Care Oil,Unani,2165
4,100005,C000007,2025-01-13,Joint Care Oil,Unani,1370


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35180 entries, 0 to 35179
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Order_ID     35180 non-null  int64 
 1   Customer_ID  35180 non-null  object
 2   Order_Date   35180 non-null  object
 3   Product      35180 non-null  object
 4   Category     35180 non-null  object
 5   Revenue      35180 non-null  int64 
dtypes: int64(2), object(4)
memory usage: 1.6+ MB


In [ ]:
df.describe()

,Order_ID,Revenue
count,35180.000000,35180.000000
mean,117590.500000,1400.258329
std,10155.735572,635.818021
min,100001.000000,300.000000
25%,108795.750000,850.000000
50%,117590.500000,1403.000000
75%,126385.250000,1950.000000
max,135180.000000,2500.000000


In [ ]:
df.isnull().sum()

,0
Order_ID,0
Customer_ID,0
Order_Date,0
Product,0
Category,0
Revenue,0


In [ ]:
first_purchase = df.groupby('Customer_ID')['Order_Date'].min().reset_index()
first_purchase.columns = ['Customer_ID' , 'First_Purchase_Date']
first_purchase['Cohort_Month'] = (first_purchase['First_Purchase_Date'].dt.to_period('M'))
first_purchase.head()

,Customer_ID,First_Purchase_Date,Cohort_Month
0,C000001,2025-02-01,2025-02
1,C000002,2025-01-19,2025-01
2,C000003,2025-01-14,2025-01
3,C000004,2025-02-22,2025-02
4,C000005,2025-01-23,2025-01


In [ ]:
# STEP 2: Add cohort info to every transaction

df = df.merge(first_purchase['Customer_ID'] , on = 'Customer_ID')
df['Txn_Month'] = df['Order_Date'].dt.to_period('M')
df['Month_Offset'] = (df['Txn_Month'] - df['Cohort_Month']).apply(lambda x: x.n)
df['Is_NTB'] = (df['Month_Offset'] == 0).astype(int)

In [ ]:
# STEP 3: Save enriched file

df.to_csv('cohort_enriched.csv' , index = False)

In [ ]:
# STEP 4: Build retention matrix

# STEP 4: Build retention matrix
cohort_sizes = first_purchase.groupby('Cohort_Month').size().reset_index(name='NTB_Count')

retention = (df[df['Month_Offset'] > 0]
             .groupby(['Cohort_Month','Month_Offset'])['Customer_ID']
             .nunique()
             .reset_index())
retention.columns = ['Cohort_Month','Month_Offset','Returning_Customers']
retention = retention.merge(cohort_sizes, on='Cohort_Month')
retention['Retention_Pct'] = (retention['Returning_Customers'] /
                               retention['NTB_Count'] * 100).round(1)

In [ ]:
# STEP 5: Save matrix file
retention.to_csv('cohort_matrix.csv', index=False)